In [13]:
import json
import random
from pathlib import Path

PRODUCTION_DATA = Path("../monitoring/production_students.json")

# limpa arquivo
if PRODUCTION_DATA.exists():
    PRODUCTION_DATA.unlink()

PRODUCTION_DATA.parent.mkdir(exist_ok=True)

def generate_student():

    return {
        "IDADE_ALUNO": random.randint(11, 22),
        "ANO_INGRESSO": random.randint(2018, 2024),
        "CG": random.uniform(0, 10),
        "CF": random.uniform(0, 10),
        "CT": random.uniform(0, 10),
        "QTDE_AVAL": random.randint(1, 6),
        "IAA": random.uniform(0, 10),
        "IEG": random.uniform(0, 10),
        "IPS": random.uniform(0, 10),
        "NOTA_MAT": random.uniform(0, 10),
        "NOTA_PORT": random.uniform(0, 10),
        "NOTA_ING": random.uniform(0, 10),
        "IPV": random.uniform(0, 10),
        "IPP": random.uniform(0, 10),
        "PEDRA_ORDINAL": random.randint(1, 4),
        "NOTA_MEDIA": random.uniform(0, 10),
        "TEMPO_PM": random.randint(0, 10),
        "FASE": random.choice(["1", "2", "3"]),
        "TURMA": random.choice(["A", "B", "C"])
    }

students = [generate_student() for _ in range(100)]

with open(PRODUCTION_DATA, "w", encoding="utf-8") as f:
    json.dump(students, f, indent=2)

print("100 alunos gerados com sucesso!")

100 alunos gerados com sucesso!


In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

REFERENCE_PATH = Path("../monitoring/reference.csv")
OUTPUT_PATH = Path("../monitoring/production.csv")

# -------------------------------
# carregar reference
# -------------------------------

reference = pd.read_csv(REFERENCE_PATH)

if "DEFASADO" in reference.columns:
    reference = reference.drop(columns=["DEFASADO"])

# manter apenas features usadas no modelo
cat_features = [c for c in reference.columns if c.startswith("CAT_")]

reference = reference[cat_features]

print("Reference shape:", reference.shape)

# -------------------------------
# criar base production
# -------------------------------

production = reference.sample(n=1000, replace=True).copy()

# -------------------------------
# aplicar drift controlado
# -------------------------------

drift_columns = [
    "CAT_NOTA_MEDIA",
    "CAT_IDADE_ALUNO",
    "CAT_QTDE_AVAL"
]

drift_ratio = 0.2
n_drift = int(len(production) * drift_ratio)

drift_idx = np.random.choice(production.index, n_drift, replace=False)

for col in drift_columns:

    # desloca categoria
    production.loc[drift_idx, col] = (
        production.loc[drift_idx, col] + 1
    )

print("Drift aplicado em:", drift_columns)
print("Linhas com drift:", n_drift)

# -------------------------------
# salvar production
# -------------------------------

production.to_csv(OUTPUT_PATH, index=False)

print("\nProduction dataset salvo em:")
print(OUTPUT_PATH)

Reference shape: (1874, 19)
Drift aplicado em: ['CAT_NOTA_MEDIA', 'CAT_IDADE_ALUNO', 'CAT_QTDE_AVAL']
Linhas com drift: 200

Production dataset salvo em:
..\monitoring\production.csv
